# Anticipate a deadline

A proactive agent pursues a standing goal as time and circumstances change. This notebook uses synthetic meetings and local tools. No model calls, credentials or external accounts are used.

Install the repository and `examples/proactive/requirements.txt` in your kernel environment. See [the tutorial](../README.md) for the conceptual explanation, diagrams, citations and limitations.

In [1]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/noema').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
from examples.proactive.lessons import run_lesson

level = 1

## Read the actual policy

These functions are shared business code, not framework-provided behavior. Source is displayed so a small invocation does not hide its implementation.

In [3]:
import inspect

from examples.proactive.domain import Tools, proposals

print(inspect.getsource(proposals))
print(inspect.getsource(Tools.invoke))

def proposals(world):
    """Anticipate a preparation gap; do nothing outside the useful window."""
    if not world["enabled"]:
        return []
    candidates = []
    for name, meeting in world["meetings"].items():
        remaining = meeting["start"] - world["now"]
        revision = meeting["revision"]
        version = f"{name}:v{revision}"
        if meeting.get("cancelled") or not 0 < remaining <= 5:
            continue
        if version in world["notified"]:
            continue
        if world["level"] == 1:
            tool, reason = "notify", "Preparation is due before the meeting"
        elif version not in world["notes"]:
            if world["reads"].get(version) == world["now"]:
                continue  # Unavailable evidence is not evidence of absence.
            tool, reason = "read_note", "Resolve a missing preparation fact"
        elif version not in world["drafts"]:
            tool, reason = "prepare_brief", "Prepare from the current note"
        else:
  

## Run the Noema implementation

The scenario asserts useful action and appropriate silence.

In [4]:
from IPython.display import display

result = await run_lesson('noema', level)
display(result['calls'])
display(result['notices'])
assert len(result['notices']) == 1

[{'tool': 'notify', 'version': 'design:v1', 'at': 5}]

[{'version': 'design:v1',
  'at': 5,
  'text': 'Prepare for your upcoming meeting'}]

## Inspect the SDK bridge

NoemaLesson is a tutorial adapter, not an SDK API. Its source uses real ActionIntent, AutonomousAgent, NoemaSystem, policy and capability APIs.

In [5]:
from examples.proactive.noema_agent import NoemaLesson

print(inspect.getsource(NoemaLesson))

class NoemaLesson:
    def __init__(self, level, tools, database=None):
        self.level, self.tools = level, tools
        self.kernel = NoemaKernel(store=SQLiteEventStore(database)) if database else NoemaKernel()
        registry = CapabilityRegistry()

        def execute(arguments, context):
            proposal = Proposal(**arguments)
            world = context.situation.fact("world")
            updated = tools.invoke(proposal, world)
            return CapabilityResult.ok({"tool": proposal.tool}, facts={"world": updated})

        for name in ("read_note", "prepare_brief", "notify"):
            registry.register_function(CapabilitySpec(name, name.replace("_", " ")), execute)

        def reason(request):
            world = request.situation.fact("world")
            if world is None:
                return []
            return [
                ActionIntent(
                    candidate.tool,
                    candidate.payload(),
                    rationale=candidate

## Reproduce with ADK and LangGraph

Each equivalent executes its own framework. Compare tool calls, notices and final state, not incompatible raw event formats.

In [6]:
equivalents = {}
for framework in ('adk', 'langgraph'):
    other = await run_lesson(framework, level)
    for key in ('calls', 'notices', 'world'):
        assert other[key] == result[key], (framework, key)
    equivalents[framework] = {'calls': len(other['calls']), 'notices': len(other['notices'])}
display(equivalents)

{'adk': {'calls': 1, 'notices': 1}, 'langgraph': {'calls': 1, 'notices': 1}}

## Change the circumstances

Run an independent case where a meeting is cancelled before its window. Change `cancelled` to False to observe how that changes eligibility.

In [7]:
tools = Tools({'design:v1': 'A synthetic agenda'})
agent = await NoemaLesson(level, tools).start()
cancelled = {'start': 10, 'revision': 1, 'importance': 5, 'cancelled': True}
try:
    await agent.tick(now=4, meetings={'design': cancelled})
    await agent.tick(now=5)
    assert tools.notices == []
    display(tools.calls)
finally:
    await agent.close()

[]